# Multi-Stage Training Pipeline — Registry-Backed Lineage

Real production ML pipelines rarely fit into one training run.
PhysicsX's own workflow looks like this: a **platform team**
publishes the model architecture and curated datasets, an **init
team** does a cheap warm-up on synthetic data, a **pretraining team**
scales training across a broad flight envelope, and a **posttraining
team** fine-tunes on a narrow high-value regime. Each team hands off
to the next through the **W&B Registry** — versioned, aliased,
lineage-tracked model and dataset artifacts.

This notebook demonstrates the three training stages against the
workshop's `TinyTransolver`:

- **§3 Stage 1 — the init team** consumes `airfoil_models/base` +
  `airfoil_datasets/synthetic`, promotes to
  `airfoil_models/initialised`.
- **§4 Stage 2 — the pretraining team** consumes
  `airfoil_models/initialised` + `airfoil_datasets/pretraining`,
  promotes to `airfoil_models/pretrained`.
- **§5 Stage 3 — the posttraining team** consumes
  `airfoil_models/pretrained` + `airfoil_datasets/posttraining`,
  promotes to `airfoil_models/posttrained` — the final production
  model.
- **§6** walks the resulting lineage graph programmatically.
- **§7** discusses how the pattern extends to multiple teams,
  multiple architectures, and parallel experiment lineages.

**Prerequisites**
- `uv sync`, `uv run wandb login`.
- **`notebooks/advanced_multistage_setup.ipynb` executed first** — that
  notebook provisions the base model + three datasets this one
  consumes. All Registry paths below assume those entries exist at
  `:production`.
- Access to the `wandb_emea` entity — or edit `ENTITY` in §1.


## 1. Setup

Third-party imports and constants. Both Registry names are declared
here so every stage below references them consistently.


In [ ]:
from __future__ import annotations

import json
import math
import tempfile
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb

device = torch.device("cpu")

ENTITY = "wandb_emea"
PROJECT = "jb_physics_demo"

DATASETS_REG = "airfoil_datasets"
MODELS_REG = "airfoil_models"

VAL_FRACTION = 0.2


def utc_ts() -> str:
    return datetime.now(timezone.utc).isoformat()


print(f"torch      = {torch.__version__}")
print(f"wandb      = {wandb.__version__}")
print(f"Registries = {DATASETS_REG}, {MODELS_REG}")


## 2. Model + consumer utilities

The three-stage pipeline **consumes** Registry entries — it doesn't
generate anything new (that's `advanced_multistage_setup.ipynb`'s job). So
this cell inlines only what the three stage teams need:

- `TinyTransolver` — the shared architecture. Same tensor shapes at
  every stage make `state_dict` transfer sound.
- `AirfoilCase` — dataclass for typed-loading the Registry-served
  JSON datasets.
- `load_cases_from_json_path`, `stack_cases` — read the dataset
  payload into tensors.
- `train_one_stage` — the small training loop each of §3/§4/§5
  calls.

Read this once and skim past — the interesting story is in §3.


In [ ]:
# ------------------- Model -------------------
class PhysicsAttention(nn.Module):
    def __init__(self, hidden: int, n_slices: int, n_heads: int, ffn_expansion: int = 2):
        super().__init__()
        assert hidden % n_heads == 0
        self.n_slices = n_slices
        self.n_heads = n_heads
        self.head_dim = hidden // n_heads

        self.slice_proj = nn.Linear(hidden, n_slices)
        self.qkv = nn.Linear(hidden, 3 * hidden)
        self.attn_out = nn.Linear(hidden, hidden)
        self.ffn = nn.Sequential(
            nn.Linear(hidden, ffn_expansion * hidden),
            nn.GELU(),
            nn.Linear(ffn_expansion * hidden, hidden),
        )
        self.norm1 = nn.LayerNorm(hidden)
        self.norm2 = nn.LayerNorm(hidden)

    def _slice_attend_deslice(self, x: torch.Tensor) -> torch.Tensor:
        B, N, C = x.shape
        M, H, D = self.n_slices, self.n_heads, self.head_dim
        w = torch.softmax(self.slice_proj(x), dim=-1)
        numer = torch.einsum("bnm,bnc->bmc", w, x)
        denom = w.sum(dim=1).unsqueeze(-1) + 1e-8
        z = numer / denom
        qkv = self.qkv(z).view(B, M, 3, H, D).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(D)
        attn = torch.softmax(scores, dim=-1)
        z_out = (attn @ v).transpose(1, 2).contiguous().view(B, M, C)
        z_out = self.attn_out(z_out)
        return torch.einsum("bnm,bmc->bnc", w, z_out)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self._slice_attend_deslice(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x


class TinyTransolver(nn.Module):
    def __init__(
        self,
        in_dim: int = 5, out_dim: int = 3, hidden: int = 128,
        n_layers: int = 6, n_slices: int = 32, n_heads: int = 8,
        ffn_expansion: int = 2,
    ):
        super().__init__()
        self.embed = nn.Linear(in_dim, hidden)
        self.blocks = nn.ModuleList([
            PhysicsAttention(hidden, n_slices, n_heads, ffn_expansion)
            for _ in range(n_layers)
        ])
        self.out_norm = nn.LayerNorm(hidden)
        self.head = nn.Linear(hidden, out_dim)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        x = self.embed(features)
        for block in self.blocks:
            x = block(x)
        return self.head(self.out_norm(x))


MODEL_CONFIG = dict(
    in_dim=5, out_dim=3, hidden=128, n_layers=6,
    n_slices=32, n_heads=8, ffn_expansion=2,
)


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ------------------- Data (consumer side) -------------------
@dataclass
class AirfoilCase:
    points: np.ndarray
    sdf: np.ndarray
    aoa: float
    re: float
    pressure: np.ndarray
    velocity: np.ndarray

    def features(self) -> np.ndarray:
        n = self.points.shape[0]
        aoa_col = np.full((n, 1), self.aoa, dtype=np.float32)
        re_col = np.full((n, 1), np.log10(self.re) - 6.0, dtype=np.float32)
        return np.concatenate(
            [self.points.astype(np.float32),
             self.sdf[:, None].astype(np.float32),
             aoa_col, re_col],
            axis=1,
        )

    def targets(self) -> np.ndarray:
        return np.concatenate(
            [self.pressure[:, None], self.velocity], axis=1
        ).astype(np.float32)


def load_cases_from_json_path(path: Path) -> tuple[list[AirfoilCase], dict]:
    payload = json.loads(path.read_text())
    cases = [
        AirfoilCase(
            points=np.array(c["points"], dtype=np.float32),
            sdf=np.array(c["sdf"], dtype=np.float32),
            aoa=float(c["aoa"]), re=float(c["re"]),
            pressure=np.array(c["pressure"], dtype=np.float32),
            velocity=np.array(c["velocity"], dtype=np.float32),
        )
        for c in payload["cases"]
    ]
    return cases, payload["meta"]


def stack_cases(cases_list: list[AirfoilCase]) -> tuple[torch.Tensor, torch.Tensor]:
    X = torch.stack([torch.from_numpy(c.features()).float() for c in cases_list])
    Y = torch.stack([torch.from_numpy(c.targets()).float() for c in cases_list])
    return X, Y


# ------------------- Training loop shared across stages -------------------
def train_one_stage(
    model: nn.Module,
    X_train_n: torch.Tensor, Y_train_n: torch.Tensor,
    X_val_n: torch.Tensor, Y_val_n: torch.Tensor,
    run, epochs: int, batch_size: int, lr: float,
) -> float:
    """Train `model` for `epochs`, logging to `run`. Returns best val pressure_rmse."""
    optim = torch.optim.AdamW(model.parameters(), lr=lr)
    best_p_rmse = float("inf")
    n = X_train_n.size(0)
    for epoch in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(n)
        losses = []
        for start in range(0, n, batch_size):
            idx = perm[start:start + batch_size]
            xb, yb = X_train_n[idx], Y_train_n[idx]
            pred = model(xb)
            loss = F.mse_loss(pred, yb)
            optim.zero_grad()
            loss.backward()
            optim.step()
            losses.append(loss.item())
        train_loss = float(np.mean(losses))

        model.eval()
        with torch.no_grad():
            err = model(X_val_n) - Y_val_n
            p_rmse = err[..., 0].pow(2).mean().sqrt().item()
        best_p_rmse = min(best_p_rmse, p_rmse)
        run.log({
            "epoch": epoch,
            "train/loss": train_loss,
            "val/pressure_rmse": p_rmse,
        })
        print(f"  epoch {epoch}: train/loss={train_loss:.4f}   val/pressure_rmse={p_rmse:.4f}")
    return best_p_rmse


tmp_dir = Path(tempfile.mkdtemp())
print(f"model params = {count_params(TinyTransolver()):,}")


## 3. Stage 1 — the init team

The init team pulls `airfoil_models/base:production` (the
un-trained architecture) and `airfoil_datasets/synthetic:production`
(the small warm-up dataset). It:

1. Computes normalisation stats on the synthetic training split.
2. Trains briefly (3 epochs) to warm the weights.
3. Promotes the result to `airfoil_models/initialised:production`,
   **bundling the normalisation stats into the checkpoint metadata**
   so downstream stages read them back rather than recomputing.

Baking the input contract into the checkpoint is the pattern that
keeps a multi-team pipeline consistent — Stage 2 and Stage 3 will
consume the same `feat_mean`, `feat_std`, `tgt_mean`, `tgt_std` as
Stage 1 decided them.


In [ ]:
with wandb.init(
    entity=ENTITY, project=PROJECT,
    job_type="stage1-init",
    name="stage1-init-team",
    tags=["notebook", "multistage", "init"],
) as run:
    # ---- Consume Registry inputs (creates lineage edges) ----
    base_art = run.use_artifact(f"wandb-registry-{MODELS_REG}/base:production")
    ds_art = run.use_artifact(f"wandb-registry-{DATASETS_REG}/synthetic:production")

    # ---- Load base model ----
    base_ckpt = torch.load(
        next(Path(base_art.download()).glob("*.pt")),
        map_location="cpu",
    )
    model = TinyTransolver(**base_ckpt["config"]).to(device)
    model.load_state_dict(base_ckpt["state_dict"])

    # ---- Load synthetic dataset ----
    cases, ds_meta = load_cases_from_json_path(
        next(Path(ds_art.download()).glob("*.json"))
    )
    n_val = max(1, int(len(cases) * VAL_FRACTION))
    train_cases, val_cases = cases[:-n_val], cases[-n_val:]
    X_train, Y_train = stack_cases(train_cases)
    X_val, Y_val = stack_cases(val_cases)

    # ---- Stage 1 computes fresh normalisation stats ----
    FEAT_MEAN = X_train.mean(dim=(0, 1), keepdim=True)
    FEAT_STD = X_train.std(dim=(0, 1), keepdim=True).clamp(min=1e-6)
    TGT_MEAN = Y_train.mean(dim=(0, 1), keepdim=True)
    TGT_STD = Y_train.std(dim=(0, 1), keepdim=True).clamp(min=1e-6)

    X_TRAIN_N = ((X_train - FEAT_MEAN) / FEAT_STD).to(device)
    Y_TRAIN_N = ((Y_train - TGT_MEAN) / TGT_STD).to(device)
    X_VAL_N = ((X_val - FEAT_MEAN) / FEAT_STD).to(device)
    Y_VAL_N = ((Y_val - TGT_MEAN) / TGT_STD).to(device)

    print(f"[stage1] loaded {len(cases)} synthetic cases; training {len(train_cases)}, "
          f"val {len(val_cases)}")

    # ---- Train ----
    best_p_rmse = train_one_stage(
        model, X_TRAIN_N, Y_TRAIN_N, X_VAL_N, Y_VAL_N,
        run=run, epochs=3, batch_size=8, lr=3e-3,
    )

    # ---- Save + promote ----
    init_path = tmp_dir / "stage1_initialised.pt"
    torch.save(
        {
            "state_dict": model.state_dict(),
            "config": dict(MODEL_CONFIG),
            "stage": "initialised",
            "feat_mean": FEAT_MEAN.squeeze().tolist(),
            "feat_std": FEAT_STD.squeeze().tolist(),
            "tgt_mean": TGT_MEAN.squeeze().tolist(),
            "tgt_std": TGT_STD.squeeze().tolist(),
            "val_pressure_rmse": best_p_rmse,
        },
        init_path,
    )
    init_art = wandb.Artifact(
        name="airfoil_model_initialised",
        type="model",
        description="Stage 1 output — warm-initialised on the synthetic dataset.",
        metadata={
            "stage": "initialised",
            "source_base": base_art.name,
            "source_dataset": ds_art.name,
            "val_pressure_rmse": best_p_rmse,
            **MODEL_CONFIG,
            "created_at": utc_ts(),
        },
    )
    init_art.add_file(str(init_path), name="checkpoint.pt")
    logged = run.log_artifact(init_art)
    logged.wait()
    run.link_artifact(
        logged,
        target_path=f"wandb-registry-{MODELS_REG}/initialised",
        aliases=["production", "v1"],
    )
    run.summary.update({
        "stage": "initialised",
        "val/pressure_rmse": best_p_rmse,
    })
    print(f"[stage1] best val/pressure_rmse={best_p_rmse:.4f}   "
          f"-> wandb-registry-{MODELS_REG}/initialised:production")


## 4. Stage 2 — the pretraining team

The pretraining team pulls
`airfoil_models/initialised:production` and
`airfoil_datasets/pretraining:production`. It reads the
normalisation stats **from the loaded checkpoint** — Stage 1
"decided" the model's input contract, and downstream stages honour
it. Trains on the broad pretraining distribution and promotes to
`airfoil_models/pretrained:production`.


In [ ]:
with wandb.init(
    entity=ENTITY, project=PROJECT,
    job_type="stage2-pretrain",
    name="stage2-pretraining-team",
    tags=["notebook", "multistage", "pretrain"],
) as run:
    src_model_art = run.use_artifact(
        f"wandb-registry-{MODELS_REG}/initialised:production"
    )
    src_ds_art = run.use_artifact(
        f"wandb-registry-{DATASETS_REG}/pretraining:production"
    )

    src_ckpt = torch.load(
        next(Path(src_model_art.download()).glob("*.pt")),
        map_location="cpu",
    )
    model = TinyTransolver(**src_ckpt["config"]).to(device)
    model.load_state_dict(src_ckpt["state_dict"])

    cases, ds_meta = load_cases_from_json_path(
        next(Path(src_ds_art.download()).glob("*.json"))
    )
    n_val = max(1, int(len(cases) * VAL_FRACTION))
    train_cases, val_cases = cases[:-n_val], cases[-n_val:]
    X_train, Y_train = stack_cases(train_cases)
    X_val, Y_val = stack_cases(val_cases)

    # ---- Reuse Stage 1's normalisation stats (they travel with the model) ----
    FEAT_MEAN = torch.tensor(src_ckpt["feat_mean"]).view(1, 1, -1)
    FEAT_STD = torch.tensor(src_ckpt["feat_std"]).view(1, 1, -1)
    TGT_MEAN = torch.tensor(src_ckpt["tgt_mean"]).view(1, 1, -1)
    TGT_STD = torch.tensor(src_ckpt["tgt_std"]).view(1, 1, -1)
    X_TRAIN_N = ((X_train - FEAT_MEAN) / FEAT_STD).to(device)
    Y_TRAIN_N = ((Y_train - TGT_MEAN) / TGT_STD).to(device)
    X_VAL_N = ((X_val - FEAT_MEAN) / FEAT_STD).to(device)
    Y_VAL_N = ((Y_val - TGT_MEAN) / TGT_STD).to(device)

    print(f"[stage2] loaded {len(cases)} pretraining cases; training {len(train_cases)}, "
          f"val {len(val_cases)}")

    best_p_rmse = train_one_stage(
        model, X_TRAIN_N, Y_TRAIN_N, X_VAL_N, Y_VAL_N,
        run=run, epochs=3, batch_size=16, lr=1e-3,
    )

    pretrained_path = tmp_dir / "stage2_pretrained.pt"
    torch.save(
        {
            "state_dict": model.state_dict(),
            "config": dict(MODEL_CONFIG),
            "stage": "pretrained",
            "feat_mean": src_ckpt["feat_mean"],
            "feat_std": src_ckpt["feat_std"],
            "tgt_mean": src_ckpt["tgt_mean"],
            "tgt_std": src_ckpt["tgt_std"],
            "val_pressure_rmse": best_p_rmse,
        },
        pretrained_path,
    )
    pretrained_art = wandb.Artifact(
        name="airfoil_model_pretrained",
        type="model",
        description="Stage 2 output — pretrained on the broad flight envelope.",
        metadata={
            "stage": "pretrained",
            "source_model": src_model_art.name,
            "source_dataset": src_ds_art.name,
            "val_pressure_rmse": best_p_rmse,
            **MODEL_CONFIG,
            "created_at": utc_ts(),
        },
    )
    pretrained_art.add_file(str(pretrained_path), name="checkpoint.pt")
    logged = run.log_artifact(pretrained_art)
    logged.wait()
    run.link_artifact(
        logged,
        target_path=f"wandb-registry-{MODELS_REG}/pretrained",
        aliases=["production", "v1"],
    )
    run.summary.update({
        "stage": "pretrained",
        "val/pressure_rmse": best_p_rmse,
    })
    print(f"[stage2] best val/pressure_rmse={best_p_rmse:.4f}   "
          f"-> wandb-registry-{MODELS_REG}/pretrained:production")


## 5. Stage 3 — the posttraining team

The posttraining team pulls `airfoil_models/pretrained:production`
and `airfoil_datasets/posttraining:production` (the narrow
high-lift regime). Same normalisation stats — still Stage 1's,
propagated through Stage 2. This is the **final production model**,
promoted to `airfoil_models/posttrained:production`.


In [ ]:
with wandb.init(
    entity=ENTITY, project=PROJECT,
    job_type="stage3-posttrain",
    name="stage3-posttraining-team",
    tags=["notebook", "multistage", "posttrain"],
) as run:
    src_model_art = run.use_artifact(
        f"wandb-registry-{MODELS_REG}/pretrained:production"
    )
    src_ds_art = run.use_artifact(
        f"wandb-registry-{DATASETS_REG}/posttraining:production"
    )

    src_ckpt = torch.load(
        next(Path(src_model_art.download()).glob("*.pt")),
        map_location="cpu",
    )
    model = TinyTransolver(**src_ckpt["config"]).to(device)
    model.load_state_dict(src_ckpt["state_dict"])

    cases, ds_meta = load_cases_from_json_path(
        next(Path(src_ds_art.download()).glob("*.json"))
    )
    n_val = max(1, int(len(cases) * VAL_FRACTION))
    train_cases, val_cases = cases[:-n_val], cases[-n_val:]
    X_train, Y_train = stack_cases(train_cases)
    X_val, Y_val = stack_cases(val_cases)

    # Norm stats still inherited from Stage 1 via the checkpoint chain.
    FEAT_MEAN = torch.tensor(src_ckpt["feat_mean"]).view(1, 1, -1)
    FEAT_STD = torch.tensor(src_ckpt["feat_std"]).view(1, 1, -1)
    TGT_MEAN = torch.tensor(src_ckpt["tgt_mean"]).view(1, 1, -1)
    TGT_STD = torch.tensor(src_ckpt["tgt_std"]).view(1, 1, -1)
    X_TRAIN_N = ((X_train - FEAT_MEAN) / FEAT_STD).to(device)
    Y_TRAIN_N = ((Y_train - TGT_MEAN) / TGT_STD).to(device)
    X_VAL_N = ((X_val - FEAT_MEAN) / FEAT_STD).to(device)
    Y_VAL_N = ((Y_val - TGT_MEAN) / TGT_STD).to(device)

    print(f"[stage3] loaded {len(cases)} posttraining cases; training {len(train_cases)}, "
          f"val {len(val_cases)}")

    best_p_rmse = train_one_stage(
        model, X_TRAIN_N, Y_TRAIN_N, X_VAL_N, Y_VAL_N,
        run=run, epochs=3, batch_size=8, lr=5e-4,
    )

    posttrained_path = tmp_dir / "stage3_posttrained.pt"
    torch.save(
        {
            "state_dict": model.state_dict(),
            "config": dict(MODEL_CONFIG),
            "stage": "posttrained",
            "feat_mean": src_ckpt["feat_mean"],
            "feat_std": src_ckpt["feat_std"],
            "tgt_mean": src_ckpt["tgt_mean"],
            "tgt_std": src_ckpt["tgt_std"],
            "val_pressure_rmse": best_p_rmse,
        },
        posttrained_path,
    )
    posttrained_art = wandb.Artifact(
        name="airfoil_model_posttrained",
        type="model",
        description="Stage 3 output — final production model, fine-tuned on high-lift regime.",
        metadata={
            "stage": "posttrained",
            "source_model": src_model_art.name,
            "source_dataset": src_ds_art.name,
            "val_pressure_rmse": best_p_rmse,
            **MODEL_CONFIG,
            "created_at": utc_ts(),
        },
    )
    posttrained_art.add_file(str(posttrained_path), name="checkpoint.pt")
    logged = run.log_artifact(posttrained_art)
    logged.wait()
    run.link_artifact(
        logged,
        target_path=f"wandb-registry-{MODELS_REG}/posttrained",
        aliases=["production", "v1"],
    )
    run.summary.update({
        "stage": "posttrained",
        "val/pressure_rmse": best_p_rmse,
    })
    print(f"[stage3] best val/pressure_rmse={best_p_rmse:.4f}   "
          f"-> wandb-registry-{MODELS_REG}/posttrained:production   (FINAL)")


## 6. Trace the lineage — from posttrained back to source

This is the property that makes multi-team pipelines **auditable**.
Given only the final artifact reference, we walk backwards through
the graph via `artifact.logged_by()` (the producing run) and
`run.used_artifacts()` (that run's inputs). Repeat recursively until
you hit the leaves (base model + datasets).

In production a compliance team can start from a shipped model,
walk this tree, and reconstruct the entire chain of data assets and
model transformations that produced it — no external state, no
extra database, no messaging between teams.


In [ ]:
def print_lineage(artifact_path: str, depth: int = 0, seen: set | None = None) -> None:
    if seen is None:
        seen = set()
    api = wandb.Api()
    art = api.artifact(artifact_path)
    key = f"{art.name}:{art.version}"
    indent = "  " * depth
    stage = (art.metadata or {}).get("stage", "?")
    print(f"{indent}└─ {art.name}:{art.version}   type={art.type}   stage={stage}")

    if key in seen:
        print(f"{indent}   (already visited)")
        return
    seen.add(key)

    producer = art.logged_by()
    if producer is None:
        return
    print(f"{indent}   ↑ produced by run {producer.id}   "
          f"(job_type={producer.job_type}   name={producer.name})")
    for used in producer.used_artifacts():
        print_lineage(
            f"{used.entity}/{used.project}/{used.name}:{used.version}",
            depth + 1, seen,
        )


print(f"walking lineage from wandb-registry-{MODELS_REG}/posttrained:production")
print()
print_lineage(f"wandb-registry-{MODELS_REG}/posttrained:production")


## 7. Scaling out — multiple teams, multiple architectures

The pipeline above is intentionally single-lineage for clarity. In
production it scales in three orthogonal directions with **no code
changes** to the stage cells — only the source/target Collection
names differ.

### Multiple teams sharing one Registry

Each of §3/§4/§5 is an independent job. In production the init
team, the pretraining team, and the posttraining team run their
respective cells from *different* laptops or CI environments,
possibly on different weeks. Nothing coordinates them except the
Registry:

- Every team authenticates to W&B and runs their stage.
- The downstream team pulls `:production` (never a specific
  version), so they always get the latest promoted checkpoint.
- The moment `link_artifact` moves the `:production` pointer on an
  upstream Collection, downstream teams' next runs pick it up
  automatically.

In effect **the Registry replaces the internal messaging channel**.
No Slack pings, no "hey Bob, is the checkpoint ready?" — the Registry
*is* the ready-state signal.

Two governance features attendees should know about but are out of
scope for this demo:

- **Access control.** Registry Collections can carry ACLs at the
  org level: the posttraining team can *consume* pretrained models
  but not *overwrite* the pretrained Collection. Configure in W&B
  org admin.
- **Versioning and rollback.** Aliases *are* the release channel.
  If Team B pushes a bad `pretrained:production`, they move
  `production` back to `v1` in the UI. Downstream `use_artifact`
  calls resolve to the previous version immediately — no
  redeployment.

### Multiple model architectures in parallel

The demo assumes one shared architecture because `state_dict`
transfer requires identical tensor shapes. If two teams want to
compare architectures — say
`TinyTransolver(hidden=128, n_layers=6)` vs
`TinyTransolver(hidden=192, n_layers=4)` — the Registry
accommodates that as **parallel Collection families**:

```
airfoil_models/
  base_hidden128           ┐
  initialised_hidden128    │  Family A
  pretrained_hidden128     │  (Team A's production line)
  posttrained_hidden128    ┘
  base_hidden192           ┐
  initialised_hidden192    │  Family B
  pretrained_hidden192     │  (Team B's production line)
  posttrained_hidden192    ┘
```

Every stage cell in this notebook is parameterised only by *source*
and *target* Collection names — swap those and the same code runs
against a different architecture family. The artifact metadata
already carries `hidden`, `n_layers`, `n_slices`, etc., so
consumers can defensively check the shape they're loading (raise
an error if the checkpoint's config doesn't match the constructor
they're about to invoke). Architecture-level parallelism with zero
pipeline-code duplication — each family is just a name-prefix.

### Multiple lineages within one architecture family

Even inside one architecture family, ablations produce multiple
parallel lineages — e.g. two init runs trained from different
synthetic seeds. Re-running §3 with a different source dataset
produces `initialised:v2`. Two choices for how to expose it:

- **Move the alias**: `link_artifact(..., aliases=["production", "v2"])`
  moves the `production` pointer to the new version. All downstream
  stages next-time pick up v2.
- **Keep parallel named aliases**: `aliases=["experiment-B", "v2"]`
  keeps v1 as `production` and creates a distinct `experiment-B`
  handle. Downstream stages can consume either explicitly:
  `use_artifact(".../initialised:experiment-B")`.

Zero code change to the stage cells — the alias string is the only
knob.


## 8. Wrap up — what to look at in the W&B UI

**Registries** (browse from the org-level Registry page):
- `airfoil_datasets` with 3 Collections
  (`synthetic`, `pretraining`, `posttraining`) — each has one
  `:production`-aliased version, published by
  `advanced_multistage_setup.ipynb`.
- `airfoil_models` with 4 Collections
  (`base`, `initialised`, `pretrained`, `posttrained`) —
  `base:production` published by setup, the other three by
  §3/§4/§5 of this notebook.

**Lineage graph.** Open the `posttrained:production` artifact and
inspect the graph — it reads back through Stages 3 → 2 → 1 → base,
with each dataset attached as an input at its stage.

**Runs.** Filter the project's runs table by `job_type` to see one
team's contribution in isolation:
- `job_type=publish-base` — platform team (from
  `advanced_multistage_setup.ipynb`).
- `job_type=stage1-init` — init team.
- `job_type=stage2-pretrain` — pretraining team.
- `job_type=stage3-posttrain` — posttraining team.

Downstream consumers pull the final model at
`wandb-registry-airfoil_models/posttrained:production` — the same
API `use_artifact(...)` we used in §3/§4/§5, from any project or
process, indefinitely.
